# Lab 04 — Bayesian Cosmology with a Real Pantheon+ Likelihood

**Format:** 4-hour hands-on practical **Status:** Formative + weekly  
checkpoint **Real data:** Pantheon+ / SH0ES **Uncertainty model:** published  
statistical covariance

> ## Scientific question
>
> **How does the likelihood from Lab 03 become a posterior, and how do  
> nuisance calibration, covariance and prior choices propagate into  
> cosmological uncertainty?**

We continue with the same real-data model:

$$
m_B^{\rm corr}
=
\mu(z;\Omega_m,H_0)
+
M_B
+
\epsilon.
$$

The quantity $m_B^{\rm corr}$ is the released standardized apparent-magnitude product.  
The model predicts that quantity from the cosmological distance modulus and  
the nuisance calibration parameter $M_B$.

In this teaching exercise we fix

$$
H_0
=
70\ {\rm km\,s^{-1}\,Mpc^{-1}}
$$

only as a **zero-point convention**. We are not measuring $H_0$ in this lab.  
The main inferential object is the joint posterior

$$
p(\Omega_m,M_B\mid D),
$$

and especially the marginalized posterior

$$
p(\Omega_m\mid D).
$$

### Learning goals

By the end of the practical you should be able to explain how a covariance  
likelihood becomes a posterior, normalize a two-dimensional posterior grid,  
marginalize over $M_B$, construct credible intervals and credible regions,  
test prior sensitivity, and distinguish posterior-supported model curves from  
genuine posterior-predictive observations.


<style>
.jp-RenderedHTMLCommon,
.text_cell_render,
.markdown {
    max-width: 920px !important;
    white-space: normal !important;
    overflow-wrap: anywhere !important;
}
.jp-RenderedHTMLCommon p,
.text_cell_render p,
.markdown p {
    max-width: 920px !important;
    white-space: normal !important;
}
</style>


## 0. Before running code — the five-line statistical statement

Write these five statements before looking at the posterior. The aim is to  
state the inference problem before the numerical result influences your  
interpretation.

**Estimand:** What posterior quantity do you want to infer?

**Observed data:** Which Pantheon+ quantities enter the likelihood?

**Generative assumptions:** How do $(\Omega_m,M_B)$ generate the expected standardized  
magnitude?

**Uncertainty / selection:** Which covariance is included, and which sources  
of uncertainty are deliberately deferred?

**Validation / falsification test:** What numerical or graphical result would  
make you distrust the posterior or revisit the model?

### Student response

**Estimand:**

**Observed data:**

**Generative assumptions:**

**Uncertainty / selection:**

**Validation / falsification test:**

## 1. Imports and reproducibility

The numerical posterior grid is deterministic. The random seed is used only  
for posterior draws and bootstrap-style diagnostics later in the lab.

The code uses a Cholesky factorization of the covariance matrix. This is  
numerically preferable to forming $C^{-1}$ explicitly. Instead of computing the  
inverse matrix and then multiplying by it, we solve linear systems of the form

$$
C\mathbf x = \mathbf y.
$$

This is both more stable and more efficient.

In [ ]:
from pathlib import Path
import io, sys, time, requests, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.linalg import cho_factor, cho_solve
from scipy.interpolate import RegularGridInterpolator
from scipy.special import logsumexp
from astropy.cosmology import FlatLambdaCDM

SEED = 2026
rng = np.random.default_rng(SEED)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 2. Load the public Pantheon+ data and covariance

The Pantheon+ table is a **high-level calibrated data product**. The rows are  
not raw detector measurements.

The covariance file is stored as a flattened $N\times N$ matrix. The first number  
in the file gives $N$, followed by the $N^2$ covariance entries.

For this lab we intentionally use the released **statistical covariance**. The  
statistical+systematic covariance is reserved for Lab 05 so that the effect of  
adding systematic uncertainty can be studied explicitly.

In [ ]:
BASE = (
    "https://raw.githubusercontent.com/PantheonPlusSH0ES/"
    "DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/"
)
URL_DATA = BASE + "Pantheon%2BSH0ES.dat"
URL_STAT = BASE + "Pantheon%2BSH0ES_STATONLY.cov"
URL_FULL = BASE + "Pantheon%2BSH0ES_STAT%2BSYS.cov"

CACHE = Path("data_cache/pantheon_plus")
CACHE.mkdir(parents=True, exist_ok=True)

def download_if_needed(url, path, timeout=180):
    path = Path(path)
    if not path.exists():
        print("Downloading", path.name, "...")
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        path.write_bytes(r.content)
    return path

def load_pantheon_table():
    p = download_if_needed(URL_DATA, CACHE/"Pantheon+SH0ES.dat")
    return pd.read_csv(p, sep=r"\s+")

def load_covariance(kind="STATONLY"):
    if kind.upper() == "STATONLY":
        url = URL_STAT
        path = CACHE/"Pantheon+SH0ES_STATONLY.cov"
    elif kind.upper() in {"FULL","STAT+SYS","STATSYS"}:
        url = URL_FULL
        path = CACHE/"Pantheon+SH0ES_STAT+SYS.cov"
    else:
        raise ValueError("kind must be STATONLY or FULL")

    p = download_if_needed(url, path)
    flat = np.loadtxt(p)
    n = int(flat[0])
    values = flat[1:]
    if values.size != n*n:
        raise ValueError(f"Expected {n*n} covariance entries, found {values.size}")
    return values.reshape(n, n)

sn = load_pantheon_table()
print("Pantheon+ rows:", len(sn))

## 3. Define the teaching sample and the cosmological model

The teaching mask keeps non-Cepheid calibrators with

$$
z_{\rm HD} > 0.01.
$$

This is a simplified Hubble-flow teaching sample. It is not presented as the  
only valid sample definition for publication-level Pantheon+ cosmology.

The data table and covariance matrix must be subset with exactly the same row  
indices. If the rows and covariance are misaligned, the likelihood becomes  
mathematically invalid because the residual vector and covariance no longer  
refer to the same observations.

### Why `coefficient_grid()` has the form used in the code

For a fixed value of $\Omega_m$, define

$$
\mathbf y
=
\mathbf m_B^{\rm corr}
-
\boldsymbol\mu(\mathbf z;\Omega_m,H_0).
$$

The residual vector after including the absolute-magnitude calibration is

$$
\Delta
=
\mathbf y
-
M_B\mathbf1.
$$

The covariance chi-square is

$$
\chi^2
=
\Delta^T C^{-1}\Delta.
$$

Substitute the expression for $\Delta$:

$$
\chi^2
=
(
\mathbf y-M_B\mathbf1
)^T
C^{-1}
(
\mathbf y-M_B\mathbf1
).
$$

Expanding gives

$$
\chi^2
=
\mathbf y^T C^{-1}\mathbf y
-
2M_B
\mathbf1^T C^{-1}\mathbf y
+
M_B^2
\mathbf1^T C^{-1}\mathbf1.
$$

Define

$$
a
=
\mathbf y^T C^{-1}\mathbf y,
$$

$$
b
=
\mathbf1^T C^{-1}\mathbf y,
$$

and

$$
d
=
\mathbf1^T C^{-1}\mathbf1.
$$

Then

$$
\boxed{
\chi^2(\Omega_m,M_B)
=
a
-
2M_Bb
+
M_B^2d
}.
$$

The code therefore needs an expensive covariance solve only for each value of  
$\Omega_m$, while the dependence on $M_B$ is evaluated analytically across the  
grid.

In [ ]:
def teaching_mask(sn):
    """Real-data teaching sample: non-Cepheid calibrators with zHD > 0.01."""
    mask = (
        np.isfinite(sn["zHD"].to_numpy(float)) &
        np.isfinite(sn["m_b_corr"].to_numpy(float)) &
        (sn["zHD"].to_numpy(float) > 0.01) &
        (sn["IS_CALIBRATOR"].to_numpy(int) == 0)
    )
    return mask

def subset_data_cov(sn, cov, mask):
    idx = np.flatnonzero(mask)
    d = sn.iloc[idx].reset_index(drop=True).copy()
    c = cov[np.ix_(idx, idx)]
    return d, c, idx

def mu_flat_lcdm(z, omega_m, H0=70.0):
    cosmo = FlatLambdaCDM(H0=H0, Om0=omega_m, Tcmb0=2.725)
    return cosmo.distmod(np.asarray(z, float)).value

def coefficient_grid(data, cov, omega_grid, H0=70.0):
    """
    For model m_b_corr = mu(z; Omega_m,H0) + M_B,
    compute coefficients so that
    chi2(Omega_m, M_B) = a - 2 M_B b + M_B^2 d.
    """
    z = data["zHD"].to_numpy(float)
    m = data["m_b_corr"].to_numpy(float)
    one = np.ones(len(data))

    cf = cho_factor(cov, lower=True, check_finite=False)
    cinv_one = cho_solve(cf, one, check_finite=False)
    dcoef = one @ cinv_one

    a = np.empty(len(omega_grid))
    b = np.empty(len(omega_grid))

    for k, om in enumerate(omega_grid):
        y = m - mu_flat_lcdm(z, om, H0=H0)
        cinv_y = cho_solve(cf, y, check_finite=False)
        a[k] = y @ cinv_y
        b[k] = one @ cinv_y

    return a, b, float(dcoef)

def profile_mb(a, b, d):
    mb_hat = b/d
    chi2_prof = a - b*b/d
    return mb_hat, chi2_prof

def chi2_surface(a, b, d, mb_grid):
    mb = np.asarray(mb_grid, float)[None, :]
    return a[:,None] - 2*b[:,None]*mb + d*mb*mb

## 4. Build the two-dimensional parameter grid

The grid covers

$$
0.10
\le
\Omega_m
\le
0.60
$$

and

$$
-20.2
\le
M_B
\le
-18.2.
$$

A grid is appropriate here because the posterior has only two dimensions.  
Unlike MCMC, it lets us inspect every evaluated point directly.

Before accepting the posterior, check whether important probability reaches  
one of the grid edges. If it does, extend the grid before interpreting the  
credible region.

In [ ]:
cov_stat = load_covariance("STATONLY")
mask = teaching_mask(sn)
data, Cstat, original_indices = subset_data_cov(sn, cov_stat, mask)

OMEGA_GRID = np.linspace(0.10, 0.60, 101)
MB_GRID = np.linspace(-20.2, -18.2, 161)
H0_CONVENTION = 70.0

a, b, d = coefficient_grid(data, Cstat, OMEGA_GRID, H0=H0_CONVENTION)
CHI2 = chi2_surface(a, b, d, MB_GRID)

print("Posterior grid shape:", CHI2.shape)

# Part I — Bayes' theorem on a grid

## 1. From the Lab 03 likelihood to a posterior

For the Pantheon+ teaching model, define the residual vector

$$
\Delta
=
\mathbf m_B^{\rm corr}
-
\left[
\boldsymbol\mu(\mathbf z;\Omega_m,H_0)
+
M_B\mathbf 1
\right].
$$

With covariance matrix $C$, the Gaussian chi-square is

$$
\chi^2
=
\Delta^T C^{-1}\Delta.
$$

The full Gaussian log-likelihood contains a quadratic residual term, a  
determinant term and the usual Gaussian normalization. In this lab the  
covariance matrix is fixed, so the terms that do not depend on $(\Omega_m,M_B)$ can be  
absorbed into a constant. Therefore

$$
\boxed{
\log\mathcal L
=
-\frac12\chi^2
+
\mathrm{constant}
}.
$$

For relative posterior probabilities we can use

$$
\log\mathcal L
\propto
-\frac12\chi^2.
$$

## 2. Baseline priors

Use

$$
\Omega_m
\sim
U(0.10,0.60),
$$

and

$$
M_B
\sim
U(-20.2,-18.2).
$$

Inside those bounds the prior density is constant; outside them it is zero.  
The posterior shape inside the allowed grid is therefore proportional to the  
likelihood, but the prior still matters because it defines the support and the  
parameter boundaries.

Bayes' theorem gives

$$
p(\Omega_m,M_B\mid D)
\propto
\mathcal L(D\mid\Omega_m,M_B)
p(\Omega_m)
p(M_B).
$$

Because the baseline priors are constant on the grid, the code can normalize  
the likelihood directly to obtain the posterior.

In [ ]:
logL = -0.5 * CHI2

# Numerical normalization on the discrete grid.
logP = logL - logsumexp(logL)
P = np.exp(logP)
P /= P.sum()

print("Posterior grid normalization:", P.sum())

## 3. Plot and interpret the joint posterior

A two-dimensional posterior contains information that separate one-dimensional  
error bars cannot show. Inspect its orientation, elongation, curvature and  
distance from the prior boundaries.

A posterior contour is defined by **enclosed posterior probability**, not by  
an arbitrary fraction of the peak density. For a region $R_\alpha$,

$$
\sum_{(i,j)\in R_\alpha}
P_{ij}
\approx
\alpha.
$$

Therefore a 68% contour must be constructed so that it encloses approximately  
68% of the posterior mass.

The function below sorts grid cells from highest to lowest posterior density  
and finds the density threshold at which the desired cumulative probability is  
reached.

In [ ]:
def density_thresholds(P, probs=(0.68,0.95)):
    flat = P.ravel()
    order = np.argsort(flat)[::-1]
    sorted_p = flat[order]
    csum = np.cumsum(sorted_p)
    thresholds = []
    for q in probs:
        j = np.searchsorted(csum, q)
        thresholds.append(sorted_p[j])
    return thresholds

thr68, thr95 = density_thresholds(P, (0.68,0.95))

fig, ax = plt.subplots(figsize=(7,5))
# contour levels must be increasing:
ax.contour(MB_GRID, OMEGA_GRID, P, levels=sorted([thr95,thr68]))
ax.set_xlabel(r"$M_B$")
ax.set_ylabel(r"$\Omega_m$")
ax.set_title("Pantheon+ posterior: statistical covariance")
plt.show()

### Exercise 1 — joint posterior geometry

Describe the orientation of the $(\Omega_m,M_B)$ posterior.

Address all of the following questions:

1. Is the posterior strongly elongated, weakly correlated, or nearly  
separable?
2. What physical compensation between $\Omega_m$ and $M_B$ could produce  
the observed orientation?
3. Does important posterior probability reach a grid boundary?
4. Why would two separate “best value ± error” summaries lose information  
visible in the joint geometry?

Do not infer the answer from a correlation coefficient alone. Use the  
two-dimensional posterior itself.

### Student response

Write your interpretation here.

# Part II — Marginalization

## 4. Marginal posteriors

The joint posterior contains uncertainty in both $\Omega_m$ and $M_B$. If our  
primary scientific interest is $\Omega_m$, then $M_B$ acts as a nuisance  
calibration parameter.

Bayesian marginalization propagates uncertainty in $M_B$ by integrating it  
out:

$$
p(\Omega_m\mid D)
=
\int
p(\Omega_m,M_B\mid D)
\,dM_B.
$$

On a regular discrete grid,

$$
p(\Omega_{m,i}\mid D)
\propto
\sum_j
P_{ij}.
$$

The same operation produces $p(M_B\mid D)$ by summing over $\Omega_m$.

### Marginalization is not profiling

Lab 03 used the profile likelihood

$$
\mathcal L_{\rm prof}(\Omega_m)
=
\max_{M_B}
\mathcal L(\Omega_m,M_B).
$$

Profiling keeps only the best nuisance value at each $\Omega_m$. Marginalization  
instead averages over **all** nuisance values, weighted by posterior  
probability. Therefore a profile-likelihood MLE and a posterior median or mode  
need not be numerically identical.

In [ ]:
p_om = P.sum(axis=1)
p_mb = P.sum(axis=0)

p_om /= p_om.sum()
p_mb /= p_mb.sum()

def discrete_quantile(grid, p, q):
    c = np.cumsum(p)
    return np.interp(q, c, grid)

om_q = [discrete_quantile(OMEGA_GRID,p_om,q) for q in (0.16,0.50,0.84)]
mb_q = [discrete_quantile(MB_GRID,p_mb,q) for q in (0.16,0.50,0.84)]

print("Omega_m 16/50/84:", om_q)
print("M_B     16/50/84:", mb_q)

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(OMEGA_GRID, p_om)
ax.set_xlabel(r"$\Omega_m$")
ax.set_ylabel("Marginal posterior probability")
ax.set_title(r"$p(\Omega_m\mid D)$")
plt.show()

### Exercise 2 — posterior summaries

Report the posterior median and the central 68% credible interval for both  
$\Omega_m$ and $M_B$.

For a one-dimensional posterior $p(\theta\mid D)$, define the cumulative distribution

$$
F(\theta)
=
\int_{-\infty}^{\theta}
p(\theta'\mid D)
\,d\theta'.
$$

The central 68% credible interval is

$$
\left[
F^{-1}(0.16),
F^{-1}(0.84)
\right].
$$

Its Bayesian interpretation is that, **conditional on the adopted model,  
likelihood and prior**, 68% of posterior probability lies inside the interval.

Then compare the posterior mode in $\Omega_m$ with the profile-likelihood MLE from  
Lab 03 and explain why they need not be identical.

### Student response

**$\Omega_m$ median and 68% credible interval:**

**$M_B$ median and 68% credible interval:**

**Comparison with Lab 03:**

# Part III — Prior sensitivity

## 5. A deliberately informative alternative prior

The baseline prior allows a broad range of $\Omega_m$ values. To learn what prior  
sensitivity looks like, now apply

$$
\Omega_m
\sim
\mathcal N
(0.30,0.03^2),
$$

truncated to the grid.

This is a **teaching prior**, not a claim that this is the correct external  
prior for publication-level cosmology.

The scientific question is not merely whether the posterior changes. Ask  
**how** it changes: does the center move, does the width shrink, and how much  
of the apparent precision comes from the prior rather than from the current  
Pantheon+ likelihood?

In [ ]:
prior_om_alt = np.exp(-0.5*((OMEGA_GRID-0.30)/0.03)**2)
prior_om_alt /= prior_om_alt.sum()

log_prior2d = np.log(prior_om_alt[:,None] + 1e-300)
logP_alt = logL + log_prior2d
logP_alt -= logsumexp(logP_alt)
P_alt = np.exp(logP_alt)
P_alt /= P_alt.sum()

p_om_alt = P_alt.sum(axis=1)
p_om_alt /= p_om_alt.sum()

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(OMEGA_GRID, p_om, label="Baseline uniform prior")
ax.plot(OMEGA_GRID, p_om_alt, label="Informative teaching prior")
ax.set_xlabel(r"$\Omega_m$")
ax.set_ylabel("Marginal posterior probability")
ax.legend()
plt.show()

### Exercise 3 — prior sensitivity

Calculate the 16th, 50th and 84th percentiles under the alternative prior.

Then answer:

1. Did the posterior location change?
2. Did the posterior width change?
3. Is the likelihood strong enough to overwhelm the informative prior?
4. Which part of the tighter result comes from prior information?
5. Why is the statement “the posterior is objective” scientifically  
incomplete?

A mature Bayesian analysis does not try to hide prior influence. It asks  
whether scientifically defensible prior alternatives materially change the  
conclusion.

### Student response

Write a short paragraph distinguishing **information supplied by the data**  
from **information supplied by the prior**.

# Part IV — Posterior predictive thinking

## 6. Draw parameter pairs from the grid posterior

A posterior distribution over parameters implies a distribution over  
model-supported Hubble relations.

Draw

$$
(\Omega_m^{(s)},M_B^{(s)})
\sim
p(\Omega_m,M_B\mid D),
$$

and calculate

$$
m_B^{(s)}(z)
=
\mu
\left(
z;\Omega_m^{(s)},H_0
\right)
+
M_B^{(s)}.
$$

These curves propagate **parameter uncertainty** into observable space.

### Important distinction

The curves below are not complete posterior-predictive supernova datasets. A  
full replicated dataset would additionally draw observational scatter and  
covariance:

$$
\tilde{\mathbf m}^{(s)}
\sim
\mathcal N
\left(
\mathbf m_{\rm model}^{(s)},
C
\right).
$$

Therefore

$$
\boxed{
\text{posterior-supported mean relations}
\neq
\text{posterior-predictive observations}
}.
$$

The distinction matters because smooth model curves can be narrow even when  
future individual observations would remain substantially scattered.

In [ ]:
n_draw = 100
flat_idx = rng.choice(P.size, size=n_draw, replace=True, p=P.ravel())
io, im = np.unravel_index(flat_idx, P.shape)

z_plot = np.geomspace(max(0.01,data["zHD"].min()), data["zHD"].max(), 250)

fig, ax = plt.subplots(figsize=(8,5))
for k in range(n_draw):
    om = OMEGA_GRID[io[k]]
    mb = MB_GRID[im[k]]
    m_pred = mu_flat_lcdm(z_plot, om, H0=H0_CONVENTION) + mb
    ax.plot(z_plot, m_pred, alpha=0.05)

ax.scatter(data["zHD"], data["m_b_corr"], s=5, alpha=0.25)
ax.set_xscale("log")
ax.set_xlabel(r"$z_{\rm HD}$")
ax.set_ylabel(r"$m_B^{\rm corr}$")
ax.set_title("Posterior-supported Hubble relations over real data")
plt.show()

### Exercise 4 — residual diagnostic

Using the posterior median parameters, calculate magnitude residuals

$$
r_i
=
m_{B,i}^{\rm corr}
-
m_{B,i}^{\rm model}.
$$

Bin the residuals only for **visualization** in 6–10 redshift bins. In each  
bin calculate the median residual and a bootstrap uncertainty on that median.

The bins are not additional independent measurements and are not used in the  
likelihood. They are only a diagnostic representation of possible residual  
structure.

When you inspect the plot, ask:

1. Are the median residuals approximately consistent with zero?
2. Is there a monotonic trend with redshift?
3. Are the lowest- or highest-redshift bins unusual?
4. What part of the model would you reconsider if a strong redshift trend  
were present?
5. Why does an acceptable residual plot not prove that the cosmological  
model is uniquely correct?

### Student response

Write your interpretation here.

In [ ]:
omega_med = om_q[1]
mb_med = mb_q[1]

model_med = (
    mu_flat_lcdm(
        data["zHD"].to_numpy(float),
        omega_med,
        H0=H0_CONVENTION,
    )
    + mb_med
)

residual = (
    data["m_b_corr"].to_numpy(float)
    - model_med
)

diagnostic = data[["zHD"]].copy()
diagnostic["residual"] = residual

# TODO:
# 1. Choose 6–10 redshift bins.
# 2. Calculate the median residual in each bin.
# 3. Bootstrap the median residual in each bin.
# 4. Plot median residual versus representative redshift.
# 5. Add a horizontal line at zero.

# 7. In-class checkpoint

Before leaving the practical, show:

1. the normalized two-dimensional posterior;
2. 68% and 95% posterior-mass contours;
3. the marginalized $p(\Omega_m\mid D)$;
4. median and central 68% credible intervals for both parameters;
5. the prior-sensitivity comparison;
6. one posterior-supported Hubble-diagram visualization;
7. one residual diagnostic;
8. a short statement separating data-driven information from assumptions.

### Exit question

> Which statements in this lab are properties of the observations, and which  
> are conditional on the likelihood, covariance, prior and cosmological model?

A strong answer should explicitly mention the released data product,  
statistical covariance, flat $\Lambda$CDM assumption, fixed $H_0$ convention,  
prior support and nuisance parameter $M_B$.

# 8. Bridge to Lab 05

In two dimensions, a grid is powerful because we can evaluate and normalize  
the posterior directly and inspect its geometry without relying on a sampler.

Next week we deliberately introduce MCMC and ask whether the sampler can  
reproduce this known grid posterior.

We will then ask:

- How do we diagnose convergence?
- What are $\hat R$ and effective sample size?
- What do divergences mean?
- What changes when statistical+systematic covariance is added?
- How does nuisance-parameter geometry affect sampling?
- Why should we not use MCMC merely because it is fashionable?

The Week 4 grid posterior therefore becomes a numerical reference against  
which the Week 5 sampler can be tested.